In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# CONFIG
# ============================================================

MPING_PATH = "/glade/campaign/cisl/aiml/ai2es/winter_ptypes/ptype_qc/mPING_hourafter_interpolated_QC3.parquet"

START_VALID = "2021-12-01 00:00:00"
END_VALID   = "2022-04-01 00:00:00"

CONSENSUS_THRESHOLD = 70

PTYPE_COLS = {
    "sn_percent": "Snow",
    "ra_percent": "Rain",
    "pl_percent": "Ice Pellet",
    "fzra_percent": "Freezing Rain",
}


def prepare_mping(path, consensus_threshold=70):
    cols = [
        "datetime", "lat", "lon", "report_count",
        "sn_percent", "ra_percent", "pl_percent", "fzra_percent",
    ]

    df = pd.read_parquet(path, columns=cols)
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Actual verification period
    df = df[
        (df["datetime"] >= START_VALID) &
        (df["datetime"] < END_VALID)
    ].copy()

    # Only forecast valid times available every 6 h
    df = df[df["datetime"].dt.hour.isin([0, 6, 12, 18])].copy()

    pcols = list(PTYPE_COLS)

    # Dominant observed precipitation type
    df["obs_class"] = df[pcols].idxmax(axis=1).map(PTYPE_COLS)

    # Consensus
    df["obs_consensus"] = df[pcols].max(axis=1)

    if df["obs_consensus"].max() <= 1:
        df["obs_consensus"] *= 100

    # QC
    df = df[df["obs_consensus"] >= consensus_threshold].copy()

    # Stable ID for tracing observations later
    df = df.reset_index(drop=True)
    df["obs_id"] = np.arange(len(df), dtype=np.int64)

    return df[
        [
            "obs_id", "datetime", "lat", "lon",
            "report_count", "obs_class", "obs_consensus",
        ]
    ]

In [8]:
mping_obs = prepare_mping(
    MPING_PATH,
    consensus_threshold=CONSENSUS_THRESHOLD,
)

print(mping_obs)
print()
print(mping_obs["datetime"].min())
print(mping_obs["datetime"].max())
print(mping_obs["datetime"].dt.hour.value_counts().sort_index())
print()
print(mping_obs["obs_class"].value_counts())

       obs_id            datetime        lat         lon  report_count  \
0           0 2022-03-02 06:00:00  46.779068 -119.114014             1   
1           1 2022-03-02 06:00:00  46.315403 -119.159210             1   
2           2 2022-03-02 18:00:00  47.499931 -122.340637             1   
3           3 2022-03-02 18:00:00  46.315403 -119.159210             1   
4           4 2022-03-02 18:00:00  47.763485 -122.081787             1   
...       ...                 ...        ...         ...           ...   
30409   30409 2022-03-30 12:00:00  40.540760  -78.776520             2   
30410   30410 2022-03-30 12:00:00  41.237286  -78.665344             1   
30411   30411 2022-03-30 12:00:00  39.653549  -78.145508             1   
30412   30412 2022-03-30 12:00:00  44.964504  -93.711731             2   
30413   30413 2022-03-30 18:00:00  40.831070  -75.767853             1   

           obs_class  obs_consensus  
0               Rain          100.0  
1               Rain          100.0

In [9]:
def get_lead_hours(valid_time):
    """Return Day 1-7 lead hours for a given mPING valid time."""
    hour = pd.Timestamp(valid_time).hour

    if hour == 0:
        return np.arange(24, 169, 24)

    if hour not in (6, 12, 18):
        raise ValueError(f"Unsupported valid hour: {hour}")

    return hour + 24 * np.arange(7)

In [10]:
def get_lead_hours(valid_time):
    """Return Day 1-7 lead hours for a given mPING valid time."""
    hour = pd.Timestamp(valid_time).hour

    if hour == 0:
        return np.arange(24, 169, 24)

    if hour not in (6, 12, 18):
        raise ValueError(f"Unsupported valid hour: {hour}")

    return hour + 24 * np.arange(7)

In [11]:
def expand_forecast_days(obs):
    frames = []

    for forecast_day in range(1, 8):
        x = obs.copy()

        hour = x["datetime"].dt.hour

        x["forecast_day"] = forecast_day

        x["lead_hour"] = np.where(
            hour == 0,
            24 * forecast_day,
            hour + 24 * (forecast_day - 1),
        ).astype(int)

        x["init_time"] = (
            x["datetime"] -
            pd.to_timedelta(x["lead_hour"], unit="h")
        )

        frames.append(x)

    return pd.concat(frames, ignore_index=True)

In [12]:
matches = expand_forecast_days(mping_obs)

print(matches.shape)
display(
    matches[
        [
            "obs_id", "datetime",
            "forecast_day", "init_time", "lead_hour",
        ]
    ].head(20)
)

(212898, 10)


,obs_id,datetime,forecast_day,init_time,lead_hour
0,0,2022-03-02 06:00:00,1,2022-03-02,6
1,1,2022-03-02 06:00:00,1,2022-03-02,6
2,2,2022-03-02 18:00:00,1,2022-03-02,18
3,3,2022-03-02 18:00:00,1,2022-03-02,18
4,4,2022-03-02 18:00:00,1,2022-03-02,18
5,5,2022-03-02 18:00:00,1,2022-03-02,18
6,6,2022-03-03 00:00:00,1,2022-03-02,24
7,7,2022-03-03 00:00:00,1,2022-03-02,24
8,8,2022-03-03 00:00:00,1,2022-03-02,24
9,9,2022-03-03 00:00:00,1,2022-03-02,24


In [13]:
assert matches.groupby("obs_id").size().eq(7).all()

In [14]:
def show_time_mapping(matches, valid_time):
    valid_time = pd.Timestamp(valid_time)

    out = (
        matches[matches["datetime"] == valid_time]
        [["forecast_day", "init_time", "lead_hour"]]
        .drop_duplicates()
        .sort_values("forecast_day")
    )

    return out

In [15]:
show_time_mapping(matches, "2022-02-03 00:00:00")

,forecast_day,init_time,lead_hour
5774,1,2022-02-02,24
36188,2,2022-02-01,48
66602,3,2022-01-31,72
97016,4,2022-01-30,96
127430,5,2022-01-29,120
157844,6,2022-01-28,144
188258,7,2022-01-27,168


In [16]:
display(show_time_mapping(matches, "2022-02-03 00:00:00"))
display(show_time_mapping(matches, "2022-02-03 12:00:00"))

,forecast_day,init_time,lead_hour
5774,1,2022-02-02,24
36188,2,2022-02-01,48
66602,3,2022-01-31,72
97016,4,2022-01-30,96
127430,5,2022-01-29,120
157844,6,2022-01-28,144
188258,7,2022-01-27,168


,forecast_day,init_time,lead_hour
1450,1,2022-02-03,12
31864,2,2022-02-02,36
62278,3,2022-02-01,60
92692,4,2022-01-31,84
123106,5,2022-01-30,108
153520,6,2022-01-29,132
183934,7,2022-01-28,156


In [18]:
MODEL_CONFIG = {
    "Aurora": {
        "directory": "au_ptype",
        "prefix": "aurora",
    },
    "GraphCast": {
        "directory": "gc_ptype",
        "prefix": "graphcast",
    },
    "Pangu": {
        "directory": "pw_ptype",
        "prefix": "pangu",
    },
}

In [19]:
FORECAST_VARS = [
    "ML_ptype",
    "ML_rain", "ML_snow", "ML_icep", "ML_frzr",
    "ML_u",

    "ML_rain_ale", "ML_snow_ale",
    "ML_icep_ale", "ML_frzr_ale",

    "ML_rain_epi", "ML_snow_epi",
    "ML_icep_epi", "ML_frzr_epi",

    "BOURG_ptype",
]

In [20]:
class MPINGForecastCollocator:

    def __init__(self, base_dir, model_config=MODEL_CONFIG):
        self.base_dir = Path(base_dir)
        self.model_config = model_config


    # --------------------------------------------------------
    # Construct forecast filename
    # --------------------------------------------------------

    def forecast_path(self, model, init_time, lead_hour):
        cfg = self.model_config[model]

        init_time = pd.Timestamp(init_time)

        init_str = init_time.strftime("%Y-%m-%dT00Z")

        filename = (
            f"{cfg['prefix']}_"
            f"{init_str}_"
            f"{int(lead_hour):03d}.nc"
        )

        return (
            self.base_dir /
            cfg["directory"] /
            init_str /
            filename
        )


    # --------------------------------------------------------
    # Haversine distance
    # --------------------------------------------------------

    @staticmethod
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371.0

        lat1 = np.radians(lat1)
        lon1 = np.radians(lon1)
        lat2 = np.radians(lat2)
        lon2 = np.radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            np.sin(dlat / 2) ** 2 +
            np.cos(lat1) * np.cos(lat2) *
            np.sin(dlon / 2) ** 2
        )

        return 2 * R * np.arcsin(np.sqrt(a))


    # --------------------------------------------------------
    # Extract one valid-time group
    # --------------------------------------------------------

    def extract_group(self, ds, group):
        obs_lat = group["lat"].to_numpy()
        obs_lon = group["lon"].to_numpy()

        lat_da = xr.DataArray(obs_lat, dims="obs")
        lon_da = xr.DataArray(obs_lon, dims="obs")

        selected = ds.sel(
            latitude=lat_da,
            longitude=lon_da,
            method="nearest",
        )

        result = group.copy().reset_index(drop=True)

        result["grid_lat"] = selected["latitude"].values
        result["grid_lon"] = selected["longitude"].values

        result["grid_distance_km"] = self.haversine(
            result["lat"].values,
            result["lon"].values,
            result["grid_lat"].values,
            result["grid_lon"].values,
        )

        for var in FORECAST_VARS:
            if var in selected:
                result[var] = np.asarray(selected[var]).squeeze()

        return result


    # --------------------------------------------------------
    # Collocate one model
    # --------------------------------------------------------

    def collocate_model(self, matches, model):
        output = []

        grouped = matches.groupby(
            ["init_time", "lead_hour"],
            sort=True,
        )

        n = len(grouped)

        for i, ((init_time, lead_hour), group) in enumerate(grouped, 1):

            path = self.forecast_path(
                model,
                init_time,
                lead_hour,
            )

            if not path.exists():
                print(f"[MISSING] {path}")
                continue

            if i % 50 == 0 or i == 1:
                print(
                    f"{model}: {i}/{n} | "
                    f"{init_time} +{lead_hour:03d} h | "
                    f"Nobs={len(group)}"
                )

            with xr.open_dataset(path) as ds:
                extracted = self.extract_group(ds, group)

            extracted["model"] = model

            output.append(extracted)

        if not output:
            return pd.DataFrame()

        return pd.concat(output, ignore_index=True)


    # --------------------------------------------------------
    # Run all models
    # --------------------------------------------------------

    def run(self, matches):
        outputs = []

        for model in self.model_config:
            print("\n" + "=" * 70)
            print(model)
            print("=" * 70)

            result = self.collocate_model(
                matches,
                model,
            )

            outputs.append(result)

        return pd.concat(outputs, ignore_index=True)

In [21]:
collocator = MPINGForecastCollocator(
    "/glade/derecho/scratch/sakor/bulk/era5_init/ML_PTYPE"
)

aurora_matched = collocator.collocate_model(
    matches,
    "Aurora",
)

Aurora: 1/3332 | 2021-11-24 00:00:00 +168 h | Nobs=8


/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  with xr.open_dataset(path) as ds:
/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

Aurora: 50/3332 | 2021-11-29 00:00:00 +078 h | Nobs=10


/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  with xr.open_dataset(path) as ds:
/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

Aurora: 100/3332 | 2021-12-01 00:00:00 +072 h | Nobs=6


...

/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  with xr.open_dataset(path) as ds:
/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

Aurora: 3250/3332 | 2022-03-25 00:00:00 +138 h | Nobs=132


/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  with xr.open_dataset(path) as ds:
/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

Aurora: 3300/3332 | 2022-03-28 00:00:00 +024 h | Nobs=22


/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  with xr.open_dataset(path) as ds:
/glade/derecho/scratch/sakor/tmp/ipykernel_65120/2844516246.py:127: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's de

In [22]:
print("Rows:", f"{len(aurora_matched):,}")

print("\nUnique observations by forecast day:")
print(
    aurora_matched.groupby("forecast_day")["obs_id"]
    .nunique()
)

print("\nRows by forecast day:")
print(
    aurora_matched.groupby("forecast_day")
    .size()
)

print("\nGrid-distance QC:")
print(
    aurora_matched["grid_distance_km"]
    .describe(percentiles=[.50, .90, .95, .99])
)

print("\nMissing forecast values:")
print(
    aurora_matched[
        ["ML_ptype", "BOURG_ptype", "ML_u"]
    ].isna().sum()
)

print("\nML classes:")
print(aurora_matched["ML_ptype"].value_counts(dropna=False))

print("\nBourgouin classes:")
print(aurora_matched["BOURG_ptype"].value_counts(dropna=False))

Rows: 212,898

Unique observations by forecast day:
forecast_day
1    30414
2    30414
3    30414
4    30414
5    30414
6    30414
7    30414
Name: obs_id, dtype: int64

Rows by forecast day:
forecast_day
1    30414
2    30414
3    30414
4    30414
5    30414
6    30414
7    30414
dtype: int64

Grid-distance QC:
count    212898.000000
mean         21.020932
std          88.758488
min           0.329146
50%           9.817540
90%          14.424563
95%          15.448531
99%         582.156812
max        1113.244000
Name: grid_distance_km, dtype: float64

Missing forecast values:
ML_ptype         0
BOURG_ptype      0
ML_u           148
dtype: int64

ML classes:
ML_ptype
 1    113510
 0     90044
 3      5152
 2      4044
-1       148
Name: count, dtype: int64

Bourgouin classes:
BOURG_ptype
0    99364
1    99218
3     9934
2     4382
Name: count, dtype: int64


In [23]:
bad = (
    aurora_matched[
        aurora_matched["grid_distance_km"] > 30
    ]
    [
        [
            "obs_id", "datetime",
            "lat", "lon",
            "grid_lat", "grid_lon",
            "grid_distance_km",
        ]
    ]
    .drop_duplicates("obs_id")
    .sort_values("grid_distance_km", ascending=False)
)

print("Unique observations >30 km:", len(bad))
display(bad.head(30))

print("\nObservation coordinate ranges:")
print(
    bad[["lat", "lon"]]
    .agg(["min", "max"])
)

print("\nMatched grid coordinate ranges:")
print(
    bad[["grid_lat", "grid_lon"]]
    .agg(["min", "max"])
)

Unique observations >30 km: 578


,obs_id,datetime,lat,lon,grid_lat,grid_lon,grid_distance_km
3884,20618,2021-12-09 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
3841,15962,2021-12-09 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
20087,15085,2021-12-23 06:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
20090,20951,2021-12-23 06:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
24317,6106,2021-12-27 06:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
24332,7237,2021-12-27 06:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
219,15605,2021-12-03 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
221,20358,2021-12-03 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
423,20376,2021-12-04 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000
421,14734,2021-12-04 00:00:00,54.172417,-139.856125,50.0,-125.0,1113.244000



Observation coordinate ranges:
           lat         lon
min  17.340233 -139.856125
max  55.592899  -57.381073

Matched grid coordinate ranges:
     grid_lat  grid_lon
min      24.0    -125.0
max      50.0     -66.0


In [24]:
for threshold in [20, 25, 30, 50, 100, 250, 500]:
    n = aurora_matched.loc[
        aurora_matched["grid_distance_km"] > threshold,
        "obs_id"
    ].nunique()

    print(f">{threshold:3d} km: {n:5d} unique observations")

> 20 km:   580 unique observations
> 25 km:   578 unique observations
> 30 km:   578 unique observations
> 50 km:   578 unique observations
>100 km:   558 unique observations
>250 km:   524 unique observations
>500 km:   410 unique observations


In [25]:
invalid = aurora_matched["ML_ptype"] == -1

print("ML_ptype == -1:", invalid.sum())
print("ML_u NaN:", aurora_matched["ML_u"].isna().sum())

print(
    pd.crosstab(
        invalid,
        aurora_matched["ML_u"].isna(),
        rownames=["ML_ptype == -1"],
        colnames=["ML_u is NaN"],
    )
)

ML_ptype == -1: 148
ML_u NaN: 148
ML_u is NaN      False  True 
ML_ptype == -1               
False           212750      0
True                 0    148


In [26]:
cols = [
    "ML_ptype",
    "ML_rain", "ML_snow", "ML_icep", "ML_frzr",
    "ML_u",
    "ML_rain_ale", "ML_snow_ale", "ML_icep_ale", "ML_frzr_ale",
    "ML_rain_epi", "ML_snow_epi", "ML_icep_epi", "ML_frzr_epi",
]

display(
    aurora_matched.loc[invalid, cols].head(20)
)

,ML_ptype,ML_rain,ML_snow,ML_icep,ML_frzr,ML_u,ML_rain_ale,ML_snow_ale,ML_icep_ale,ML_frzr_ale,ML_rain_epi,ML_snow_epi,ML_icep_epi,ML_frzr_epi
3518,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3540,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19193,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19209,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19945,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19949,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43280,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43316,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47350,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47367,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
